<a href="https://colab.research.google.com/github/ArchitPandey/RAG_Pipeline/blob/main/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 52.2 MB/s eta 0:00:00


In [4]:
import pymupdf4llm
import re

markdown_txt = pymupdf4llm.to_markdown('Introduction to Agents.pdf', header=False, footer=False, ignore_images=True, write_images=False)
#print(markdown_txt)

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=11/12.
OCR on page.number=13/14.
OCR on page.number=24/25.
OCR on page.number=26/27.
OCR on page.number=27/28.
OCR on page.number=38/39.
OCR on page.number=44/45.
OCR on page.number=46/47.
OCR on page.number=47/48.
OCR on page.number=48/49.
OCR on page.number=49/50.



In [5]:
# clean md - remove image tags
markdown_txt = re.sub(
    r'<!-- Start of picture text -->.*?<!-- End of picture text -->',
    '',
    markdown_txt,
    flags=re.DOTALL
)

In [8]:
import pandas as pd

# chunking - trying recursive chunking since structural chunking cause chunk size to very big

chunk_config = [
    {
        "level": 1,
        "pattern": r"\n(?=##\s*\*\*)"
    },
    {
        "level": 2,
        "pattern": r"\n(?=###\s*\*\*)"
    },
    {
        "level": 3,
        "pattern": r"\n(?=####\s*\*\*)"
    }

]

def rec_chunker(txt, max_token, c_lvl):
  if ((len(txt)//4) <= max_token):
    return [txt]
  if (c_lvl < len(chunk_config)):
    return [txt]

  intermediate_chunks = re.split(chunk_config[c_lvl]["pattern"], txt)
  final_chunks = []
  for ic in intermediate_chunks:
    if (ic.strip()):
      final_chunks.append(rec_chunker(ic.strip(), max_token, c_lvl + 1))
  return final_chunks


chunks = rec_chunker(markdown_txt, 400, 1)

chunks_dict: list[dict] = []
for i, ch in enumerate(chunks_dict):
  dict_element = {
      "num": i+1,
      "content": ch,
      "token": len(ch) //4
  }
  chunks_dict.append(dict_element)

df = pd.DataFrame(chunks_dict)
df


""
